In [0]:
# %sql
# INSERT NEW LINES INTO dim_services_manual
# INSERT INTO prod_latam_catalog.crm_analytics.dim_services_manual (brand_code, brand_country,source_name)
# VALUES
# ('LOP', 'COL','JEBBIT'),
# ('LOP', 'PER','JEBBIT'),
# ('LOP', 'ECU','JEBBIT'),
# ('DMC', 'BRA','JEBBIT');

In [0]:
# Import packages and functions needed 
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType


# Function that identifies and explodes array and struct fields
def expand_arrays(df):
    array_columns = []
    
    # search and select the array columns
    for field in df.schema.fields:
        if isinstance(field.dataType, ArrayType):
            array_columns.append(field.name)

    # loop the array columns
    for col_name in array_columns:
        # Explode each array column
        df = df.withColumn(f"exploded_{col_name}", F.explode_outer(col_name))
        
        # Verify whether the data type of the array is a struct
        element_type = df.schema[f"exploded_{col_name}"].dataType
        if isinstance(element_type, StructType):
            # If it does, extract the struct subfields, create new fields and delete the original one
            for subfield in element_type.fieldNames():
                df = df.withColumn(f"{col_name}_{subfield}", F.col(f"exploded_{col_name}.{subfield}"))
                df = df.drop(col_name)
        else:
            # if the data type is not StructType (ie, StringType o DoubleType), replace the original column with the exploded one
            df = df.withColumn(col_name, F.col(f"exploded_{col_name}"))
        
        # delete the temporal exploded field
        df = df.drop(f"exploded_{col_name}")
    
    return df

In [0]:
# Import tables

## bmdm table
gdm = spark.table("crm_reporting.dim_gdm_brand_profile")
#print(gdm.count()) #10496032
# print(len(gdm.columns))


####### FIRST SELECTION OF ATTRIBUTES WITH AT LEAST 1 NON-VALUE
df = gdm
# Count non-null values per column
agg_expr = [F.count(F.when(F.col(c).isNotNull(), c)).alias(c) for c in df.columns]

# Execute the aggregation
non_null_counts = df.agg(*agg_expr).collect()[0]

# Keep the not enterely null columns
non_completely_null_columns = [c for c in df.columns if non_null_counts[c] > 0]

# count of not-enterely null columns
#print(len(non_completely_null_columns)) ## 77

# select the founded fields
gdm_cols_filt = df.select(non_completely_null_columns).\
  withColumnRenamed("created_dt",'created_date')



##### OTHER COLUMNS TO REMOVE
columns_to_remove = ['dept_store_regional','etl_batch_id','mdm_source','sys_created_by','beauty_supply_store']

##### Filter columns that contain the string '_dt' and the ones in columns_to_remove
columns_wt_dt = [col for col in gdm_cols_filt.columns if '_dt' not in col and col not in columns_to_remove]

# count of not-enterely null columns
#print(len(columns_wt_dt)) ## 51
#print(non_completely_null_columns)


# select the list of columns to keep
gdm_cols_filt = gdm_cols_filt.select(columns_wt_dt)
gdm_cols_filt.createOrReplaceTempView("gdm_cols_filt_vw")

In [0]:
# %sql
# select *--count(distinct brand_mdm_id) as counts
# from crm_reporting.dim_gdm_brand_profile
# where brand_mdm_id = '8473ba14591a9441ae2adf617f2af010'
#   AND brand_code = 'VIC'
#   AND brand_country = 'BRA'
# -- group by brand_mdm_id
# -- order by counts desc
# limit 10

brand_code,brand_country,mdm_source,brand_mdm_id,date_of_birth,profile_type,left_eye_color,right_eye_color,eye_correction_tool,skin_tone,skin_undertone,sun_reaction,lip_chap_dry_occasion,lips_description,face_shape,eye_shape,beauty_profile_modified_dt,analysis,analysis_modified_dt,internal_factor,external_factor,goal,conception_specificity,ingredient_limit,misc_last_modified_dt,preferred_family,preferred_note,mood,fragrance_priority,perfume_for_you,fragrance_preference_modified_dt,fragrance_routine,fragrance_routine_modified_dt,desired_color,desired_color_intensity,desired_color_finish,desired_color_permanence,desired_cover,desired_shade,hair_color_pref_modified_dt,colored,hair_length,hair_state,hair_texture,hair_type,scalp_type,dandruff_frequency,dandruff_volume,hair_loss_volume,hair_loss_start_date,hair_loss_factor,hair_density,natural_hair_color,root_color,tip_color,hair_grey_density,colorful_hair,last_hair_style_look,hair_profile_modified_dt,hair_routine_heating,heating_last_modified_dt,hair_wash_frequency,hair_wash_habit,hair_condition_frequency,hair_mask_frequency,last_hair_color_service,hair_color_service_frequency,last_hair_style_service,hair_routine,scalp_sensitivity,lowlights_highlights,hair_routine_modified_dt,desired_look,look_occasion_hair,desired_style,desired_style_finish,desired_hold,hair_style_pref_modified_dt,skincare_product_used,nail_remover_used,finger_nail_product,toes_nail_product,nail_accessory,lip_product_used,face_product_used,eye_product_used,hair_care_product_used,products_used_modified_dt,face_product_interested,eye_product_interested,products_interested_modified_dt,lip_shade_preference,lip_priorities,lip_preference_modified_dt,makeup_expected_look,makeup_motivation,makeup_priority,lash_look_priority,makeup_wear_pref,eye_skincare_use,makeup_preference_modified_dt,face_product_finish_preference,face_product_coverage_preference,lip_product_finish_preference,lip_product_format_preference,lip_shade_family_preference,nail_color_preference,nail_product_finish_preference,nail_product_coverage_preference,mascara_format,makeup_product_pref_modified_dt,makeup_removal_habit,makeup_application_location,current_look,look_influence,makeup_carried_in_bag,usual_face_makeup,usual_eye_makeup,usual_lip_makeup,usual_nail_makeup,makeup_professional_tools_purchased,concealer_use,look_occasion_makeup,makeup_routine_modified_dt,skin_sensitivity,skin_sensitivity_modified_dt,skin_type,skin_type_modified_dt,skincare_moment,dermatologist_routine,last_skincare_medical_treatment,facial_hair_grooming,last_skincare_professional_treatment,routine_goal,makeup_wear,sun_exposure_frequency,moisturizing_frequency,skin_routine_modified_dt,concern_improvement_goal,concern_improvement_modified_dt,channel_preference,channel_preference_modified_dt,created_dt,last_modified_dt,etl_batch_id,sys_created_by,sys_created_dt,retail_channel_preference,device_preference,grocery_supermarket,mass_merchandise,pure_players,drugstore,warehouse_club,dollar_channel,dept_store_national,dept_store_regional,dept_store_high_end,beauty_specialty_single_brand,beauty_specialty_multi_brand,beauty_supply_store,multi_level_marketing,tv_retail,salon_spa,derm_doctor,other,makeup_color_preference,gender,face_product_format_preference,hair_color_shade,hair_color_motivation,hair_service_location,net_promoter_score,feedback,eyebrows_shape,eyebrows_color,eyebrows_thickness,eye_crease,eyes_direction,eyes_closeness,protruding_eyes,eyelashes_length,eye_profile_modified_dt,forehead_size,nose_description,cupid_bow_visibility,mouth_width,complexion_freckled,face_profile_modified_dt,hair_color,makeup_look,category,product_category
VIC,BRA,DDM,8473ba14591a9441ae2adf617f2af010,null,null,null,null,null,null,null,null,null,null,null,null,null,"List(List(List(List(null, null, null, null, null, null, null, null)), List(List(null, null, Wrinkles, null, Texture uniformity, null, null, Whole face), List(null, null, Lack of firmness, null, Texture uniformity, null, null, CONCERN_TYPE=""Text

In [0]:
### Merged tables
merged_gdm = spark.sql("""
select 
  upper(a.source_name) as source_name,
  lower(a.registration_sub_source) AS registration_sub_source, 
  c.brand_customer_id,
  --a.created_dt as created_date,
  d.*
from prod_latam_catalog.crm_reporting.dim_customer a
inner join prod_latam_catalog.crm_analytics.dim_services_manual b
  on upper(a.source_name) = b.source_name 
  and a.brand_code = b.brand_code 
  and a.brand_country = b.brand_country
inner join prod_latam_catalog.crm_reporting.dim_customer_bridge c
  on a.source_customer_id = c.source_customer_id 
  and a.brand_code = c.brand_code 
  and a.brand_country = c.brand_country
-- inner join expld_gdm_vw d 
inner join gdm_cols_filt_vw d
  on c.brand_mdm_id = d.brand_mdm_id 
  and c.brand_code = d.brand_code 
  and c.brand_country = d.brand_country
where lower(a.registration_sub_source) IN ('skin dr','hair quiz','product finder')
  -- lower(a.registration_sub_source) not in ('contest',null,'registration','','Footer')
  AND to_date(a.created_dt) >= '2023-01-01'
group by all
order by registration_sub_source,brand_code, brand_country

""")

# print(merged_gdm.count()) #107786

107786


In [0]:
# call the expand array function twice
expld_gdm_1 = expand_arrays(merged_gdm) # explode first array levels
expld_gdm = expand_arrays(expld_gdm_1) # explode second array levels

# print(expld_gdm.count())
#print(len(expld_gdm.columns)) ## 83
# display(expld_gdm.limit(1))
# expld_gdm.createOrReplaceTempView("expld_gdm_vw")

In [0]:
# SECOND SELECTION OF ATTRIBUTES WITH AT LEAST 1 NON-VALUE

df = expld_gdm

# Count non-null values per column
agg_expr = [F.count(F.when(F.col(c).isNotNull(), c)).alias(c) for c in df.columns]

# Execute the aggregation
non_null_counts = df.agg(*agg_expr).collect()[0]

# Keep the not enterely null columns
non_completely_null_columns = [c for c in df.columns if non_null_counts[c] > 0]

# count of not-enterely null columns
#print(len(non_completely_null_columns)) ## 36

# select the list of columns to keep
def_gdm = df.select(non_completely_null_columns)
def_gdm.createOrReplaceTempView("def_gdm_vw")

In [0]:
summary = spark.sql(
    """
-- PRIMER CRUCE
select 
  source_name,
  brand_code,
  brand_country, 
  registration_sub_source,
  to_date(min(created_date)) min_date,
  to_date(max(created_date)) max_date,
  count(distinct brand_mdm_id) brand_mdm_id
from def_gdm_vw 
where lower(registration_sub_source) IN ('skin dr','hair quiz','product finder')
group by all
order by source_name,registration_sub_source,brand_code, brand_country
    """)

#print(cruce.count())
display(summary)

source_name,brand_code,brand_country,registration_sub_source,min_date,max_date,brand_mdm_id
DEMANDWARE,KER,COL,hair quiz,2022-03-28,2024-10-15,287
DEMANDWARE,KER,PER,hair quiz,2021-06-23,2024-10-15,99
DEMANDWARE,LRP,BRA,product finder,1900-01-01,2024-07-07,34960
DEMANDWARE,LRP,MEX,product finder,2016-01-12,2024-09-26,7591
DEMANDWARE,DMC,BRA,skin dr,2024-01-09,2024-09-05,1562
DEMANDWARE,LRP,BRA,skin dr,2024-10-14,2024-10-14,2
DEMANDWARE,LRP,MEX,skin dr,2022-03-19,2024-10-15,26
DEMANDWARE,VIC,BRA,skin dr,2024-07-23,2024-08-08,3
JEBBIT,KER,COL,hair quiz,2022-03-28,2024-10-15,24726
JEBBIT,KER,PER,hair quiz,2020-04-08,2024-08-20,19448


In [0]:
tmp = def_gdm.filter((F.col("registration_sub_source") == 'skin dr') &\
  (F.col("source_name") == 'DEMANDWARE') &\
  (F.col("brand_code") == 'LRP') &\
  (F.col("brand_country") == 'BRA'))

display(tmp)

source_name,registration_sub_source,brand_customer_id,brand_code,brand_country,brand_mdm_id,skin_tone,external_factor,hair_state,hair_texture,hair_type,scalp_type,hair_loss_volume,natural_hair_color,last_hair_style_look,hair_condition_frequency,last_hair_color_service,hair_color_service_frequency,scalp_sensitivity,hair_care_product_used,sun_exposure_frequency,created_date,gender,hair_color,category,product_category,hair_routine_heating_heating_tool,skin_sensitivity_skin_sensitivity,skin_sensitivity_zone,skin_type_skin_type,skin_type_zone,concern_improvement_goal_concern,channel_preference_product_purchase_channel,analysis_analysis_concern_concern,analysis_analysis_concern_concern_type,analysis_analysis_concern_zone
DEMANDWARE,skin dr,ff423462a031ef429b08b1bc5f4154eb,LRP,BRA,41931e65a01d04b168e6e6b2bc0f5deb,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2024-10-14T10:26:13Z,Female,null,null,null,null,null,null,Oily,wholeface,null,null,null,null,null
DEMANDWARE,skin dr,390d76b4721607e20188c29e7a644e29,LRP,BRA,ea52eb2ad4ce45572dd3edd97b223cd7,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2024-10-14T11:53:39Z,Female,null,null,null,null,null,null,Oily,wholeface,null,null,null,null,null
DEMANDWARE,skin dr,004b89e1d11412d79304c572d200d705,LRP,MEX,f77b58cccb66203b32113a3ed66e758c,null,null,null,null,null,null,null,null,null,null,null,null,null,null,More than 2 hours a day,2023-01-16T05:15:59Z,Female,null,null,null,null,null,null,Combination,wholeface,Deep wrinkles,null,null,null,null
DEMANDWARE,skin dr,ef5d4cfbd34a619f78b5201435906638,LRP,MEX,ec02838cdc5167298a59967ce02bdf1a,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2024-10-14T15:30:05Z,Female,null,null,null,null,null,null,Combination,wholeface,null,null,null,null,null
DEMANDWARE,skin dr,25249c0e415ea02c5b48c883411557be,LRP,MEX,6e430484b1373850e5b80c42995929c2,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2024-10-14T12:31:16Z,Female,null,null,null,null,null,null,Oily,wholeface,null,null,null,null,null
DEMANDWARE,skin dr,98b5438c1cf27411e5d09be7ed815b0e,LRP,MEX,155c39f12b0f57e7c5ead0b683577bd0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2024-10-14T13:56:34Z,Female,null,null,null,null,null,null,Dry,wholeface,null,null,null,null,null
DEMANDWARE,skin dr,f512fe8cf4842ee23b7c37f8a794724a,LRP,MEX,c7bfee45b3bfc10cd26a2ed86c099af5,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2024-10-14T14:28:48Z,Female,null,null,null,null,null,null,Oily,wholeface,null,null,null,null,null
DEMANDWARE,skin dr,69b5b600c104c7543ea299a3e0250722,LRP,MEX,e76f3b3060cd4b4de929b3f34f463a98,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2024-10-14T14:11:45Z,Female,null,null,null,null,null,null,Oily,wholeface,null,null,null,null,null
DEMANDWARE,skin dr,16eafdd6a67f22fcad648f50e8ef0db2,LRP,MEX,a1d92a5828fc181bb9adc2e0b2230cf6,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2024-10-14T13:12:35Z,Female,null,null,null,null,null,null,Combination,wholeface,null,null,null,null,null
DEMANDWARE,skin dr,83cb3fa9c13459f94970250e3ff5b66b,LRP,MEX,c05795494203b3a4988024de9e931ebe,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2024-10-14T14:47:52Z,Female,null,null,null,null,null,null,Combination,wholeface,null,null,null,null,null


In [0]:
# brand_mdm_id = '87e9fdf5145a5f20d88a46bfc39bd5a1'
# [{"analysis_clinical_sign":[{"clinical_sign":null,"normalized_score":null,"provider":null,"raw_score":null,"raw_score_range":null,"sign_range":null,"sign_type":null,"zone":null}],"analysis_concern":[{"active":null,"calculated":null,"concern":"Wrinkles","concern_range":null,"concern_type":"Texture uniformity","provider":null,"value":null,"zone":"Whole face"},{"active":null,"calculated":null,"concern":"Lack of firmness","concern_range":null,"concern_type":"Texture uniformity","provider":null,"value":null,"zone":"CONCERN_TYPE=\"Texture uniformity\";ZONE=[\"Whole face\", \"Lower part of the face\"]"},{"active":null,"calculated":null,"concern":"Eye Contour","concern_range":null,"concern_type":"Eye concern","provider":null,"value":null,"zone":"Eye"},{"active":null,"calculated":null,"concern":"Dark pigmentation","concern_range":null,"concern_type":"Color uniformity","provider":null,"value":null,"zone":"Whole face"},{"active":null,"calculated":null,"concern":"Lack of radiance","concern_range":null,"concern_type":"Color uniformity","provider":null,"value":null,"zone":"Whole face"},{"active":null,"calculated":null,"concern":"Blotchiness","concern_range":null,"concern_type":"Color uniformity","provider":null,"value":null,"zone":"Whole face"},{"active":null,"calculated":null,"concern":"Fine lines","concern_range":null,"concern_type":"Texture uniformity","provider":null,"value":null,"zone":"Whole face"}],"analysis_type":null,"calculated_age":null,"declared_age":null,"exact_age":null,"location":null,"touchpoint":null}]

In [0]:
# [{"concern":"Fine lines","expected_concern_range":null},{"concern":"Eye Contour","expected_concern_range":null},{"concern":"Wrinkles","expected_concern_range":null},{"concern":"Blotchiness","expected_concern_range":null},{"concern":"Dark pigmentation","expected_concern_range":null},{"concern":"Lack of radiance","expected_concern_range":null},{"concern":"Lack of firmness","expected_concern_range":null}]

In [0]:
%sql
select distinct *--analysis,fragrance_routine --brand_mdm_id, count(brand_mdm_id) as counts
from def_gdm_vw 
where brand_mdm_id = '8473ba14591a9441ae2adf617f2af010'

source_name,registration_sub_source,brand_customer_id,created_date,brand_code,brand_country,brand_mdm_id,skin_tone,external_factor,hair_state,hair_texture,hair_type,scalp_type,hair_loss_volume,natural_hair_color,last_hair_style_look,hair_condition_frequency,last_hair_color_service,hair_color_service_frequency,scalp_sensitivity,hair_care_product_used,sun_exposure_frequency,gender,hair_color,category,product_category,hair_routine_heating_heating_tool,skin_sensitivity_skin_sensitivity,skin_sensitivity_zone,skin_type_skin_type,skin_type_zone,concern_improvement_goal_concern,channel_preference_product_purchase_channel,analysis_analysis_concern_concern,analysis_analysis_concern_concern_type,analysis_analysis_concern_zone
DEMANDWARE,skin dr,272271605115d6aa3cc68ae6619bce9a,2024-07-23T12:39:46Z,VIC,BRA,8473ba14591a9441ae2adf617f2af010,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Normal,null,Fine lines,null,Wrinkles,Texture uniformity,Whole face
DEMANDWARE,skin dr,272271605115d6aa3cc68ae6619bce9a,2024-07-23T12:39:46Z,VIC,BRA,8473ba14591a9441ae2adf617f2af010,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Normal,null,Fine lines,null,Lack of firmness,Texture uniformity,"CONCERN_TYPE=""Texture uniformity"";ZONE=[""Whole face"", ""Lower part of the face""]"
DEMANDWARE,skin dr,272271605115d6aa3cc68ae6619bce9a,2024-07-23T12:39:46Z,VIC,BRA,8473ba14591a9441ae2adf617f2af010,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Normal,null,Fine lines,null,Eye Contour,Eye concern,Eye
DEMANDWARE,skin dr,272271605115d6aa3cc68ae6619bce9a,2024-07-23T12:39:46Z,VIC,BRA,8473ba14591a9441ae2adf617f2af010,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Normal,null,Fine lines,null,Dark pigmentation,Color uniformity,Whole face
DEMANDWARE,skin dr,272271605115d6aa3cc68ae6619bce9a,2024-07-23T12:39:46Z,VIC,BRA,8473ba14591a9441ae2adf617f2af010,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Normal,null,Fine lines,null,Lack of radiance,Color uniformity,Whole face
DEMANDWARE,skin dr,272271605115d6aa3cc68ae6619bce9a,2024-07-23T12:39:46Z,VIC,BRA,8473ba14591a9441ae2adf617f2af010,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Normal,null,Fine lines,null,Blotchiness,Color uniformity,Whole face
DEMANDWARE,skin dr,272271605115d6aa3cc68ae6619bce9a,2024-07-23T12:39:46Z,VIC,BRA,8473ba14591a9441ae2adf617f2af010,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Normal,null,Fine lines,null,Fine lines,Texture uniformity,Whole face
DEMANDWARE,skin dr,272271605115d6aa3cc68ae6619bce9a,2024-07-23T12:39:46Z,VIC,BRA,8473ba14591a9441ae2adf617f2af010,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Normal,null,Eye Contour,null,Wrinkles,Texture uniformity,Whole face
DEMANDWARE,skin dr,272271605115d6aa3cc68ae6619bce9a,2024-07-23T12:39:46Z,VIC,BRA,8473ba14591a9441ae2adf617f2af010,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Normal,null,Eye Contour,null,Lack of firmness,Texture uniformity,"CONCERN_TYPE=""Texture uniformity"";ZONE=[""Whole face"", ""Lower part of the face""]"
DEMANDWARE,skin dr,272271605115d6aa3cc68ae6619bce9a,2024-07-23T12:39:46Z,VIC,BRA,8473ba14591a9441ae2adf617f2af010,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Normal,null,Eye Contour,null,Eye Contour,Eye concern,Eye


In [0]:
%sql
select *
from crm_reporting.fact_message_tracker
where brand_mdm_id = '8473ba14591a9441ae2adf617f2af010'
  AND brand_code = 'VIC'
  AND brand_country = 'BRA'
  --AND event_year = 2024
  --AND event_month = '07'


brand_code,brand_country,event_year,event_month,source_name,source_customer_id,msg_source,msg_channel,event_type,event_dt,msg_dispatch_id,brand_mdm_id,has_email_address,iso_country_code,has_mobile_num,msg_carrier_nm,has_device_id,list_id,bounce_category,smtp_code,event_reason_dtl,batch_id,trig_send_external_key,campaign_id,campaign_type,promo_id,coupon_id,campaign_unique_id,touchpoint_num,segment_name,creative_version_id,control_test_group_name,order_num,page_url,url_link_alias,complain_site_domain,msg_from_org_nm,msg_from_address,scheduled_dispatch_dt,msg_batch_sent_dt,msg_subject,msg_context_detail,msg_def_external_key,msg_job_status,msg_preview_url,is_msg_multipart,msg_type,msg_template,msg_nm,msg_origin,transact_optin,transact_optin_dt,transact_optout_dt,mkt_optin,mkt_optin_dt,mkt_optout_dt,app_nm,app_pg_nm,app_platform_nm,app_platform_ver,geo_fence_nm,sys_token,country_name,region_name,metro_name,city_name,area_code,sys_created_dt,sys_created_by,etl_batch_id,campaign_name,program_name,global_mdm_id,address_perm_status,coupon_id_expiration_dt,creative_id,dm_reject_reason,dm_processed_dt,email_address,mobile_num


In [0]:
%sql
describe crm_reporting.fact_message_tracker

col_name,data_type,comment
brand_code,string,null
brand_country,string,null
event_year,int,null
event_month,string,null
source_name,string,null
source_customer_id,string,null
msg_source,string,null
msg_channel,string,null
event_type,string,null
event_dt,timestamp,null


In [0]:
%sql
select distinct *--analysis,fragrance_routine --brand_mdm_id, count(brand_mdm_id) as counts
from def_gdm_vw 
where brand_mdm_id = '8bedf714b5a1953065e3fceb1814022f'

source_name,registration_sub_source,brand_customer_id,brand_code,brand_country,brand_mdm_id,skin_tone,external_factor,hair_state,hair_texture,hair_type,scalp_type,hair_loss_volume,natural_hair_color,last_hair_style_look,hair_condition_frequency,last_hair_color_service,hair_color_service_frequency,scalp_sensitivity,hair_care_product_used,sun_exposure_frequency,created_date,gender,hair_color,category,product_category,hair_routine_heating_heating_tool,skin_sensitivity_skin_sensitivity,skin_sensitivity_zone,skin_type_skin_type,skin_type_zone,concern_improvement_goal_concern,channel_preference_product_purchase_channel,analysis_analysis_concern_concern,analysis_analysis_concern_concern_type,analysis_analysis_concern_zone
DEMANDWARE,hair quiz,391a0feda00cdbbe089a5d6f04ccb832,KER,COL,8bedf714b5a1953065e3fceb1814022f,null,Humidity,Wavy,Coarse,null,Very Oily,null,null,Straight,null,Straightners,null,null,Night Care,null,2024-09-19T05:01:14Z,null,null,null,null,Flat iron,null,null,null,null,Damaged,null,Damaged,Hair type,null
DEMANDWARE,hair quiz,391a0feda00cdbbe089a5d6f04ccb832,KER,COL,8bedf714b5a1953065e3fceb1814022f,null,Pollution,Wavy,Coarse,null,Very Oily,null,null,Straight,null,Straightners,null,null,Night Care,null,2024-09-19T05:01:14Z,null,null,null,null,Flat iron,null,null,null,null,Damaged,null,Damaged,Hair type,null


In [0]:
%sql
select distinct external_factor--* --analysis,concern_improvement_goal 
--brand_mdm_id, count(brand_mdm_id) as counts
from crm_reporting.dim_gdm_brand_profile
where brand_mdm_id = '8bedf714b5a1953065e3fceb1814022f'
--'8473ba14591a9441ae2adf617f2af010' #49rows

external_factor
"List(Humidity, Pollution)"


In [0]:
%sql
select distinct *--analysis,fragrance_routine --brand_mdm_id, count(brand_mdm_id) as counts
from def_gdm_vw 
where brand_mdm_id = 'b65e462063ddb916ebfeca673211b966'

source_name,registration_sub_source,brand_customer_id,created_date,brand_code,brand_country,brand_mdm_id,skin_tone,external_factor,hair_state,hair_texture,hair_type,scalp_type,hair_loss_volume,natural_hair_color,last_hair_style_look,hair_condition_frequency,last_hair_color_service,hair_color_service_frequency,scalp_sensitivity,hair_care_product_used,sun_exposure_frequency,gender,hair_color,category,product_category,hair_routine_heating_heating_tool,skin_sensitivity_skin_sensitivity,skin_sensitivity_zone,skin_type_skin_type,skin_type_zone,concern_improvement_goal_concern,channel_preference_product_purchase_channel,analysis_analysis_concern_concern,analysis_analysis_concern_concern_type,analysis_analysis_concern_zone
DEMANDWARE,skin dr,b2f555eed98db96dec348677fa1123e7,2024-08-08T09:18:40Z,VIC,BRA,b65e462063ddb916ebfeca673211b966,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Male,null,null,null,null,No,null,Normal,wholeface,Wrinkles,null,Wrinkles,Texture uniformity,Whole face
DEMANDWARE,skin dr,b2f555eed98db96dec348677fa1123e7,2024-08-08T09:18:40Z,VIC,BRA,b65e462063ddb916ebfeca673211b966,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Male,null,null,null,null,No,null,Normal,wholeface,Wrinkles,null,Lack of firmness,Texture uniformity,"CONCERN_TYPE=""Texture uniformity"";ZONE=[""Whole face"", ""Lower part of the face""]"
DEMANDWARE,skin dr,b2f555eed98db96dec348677fa1123e7,2024-08-08T09:18:40Z,VIC,BRA,b65e462063ddb916ebfeca673211b966,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Male,null,null,null,null,No,null,Normal,wholeface,Wrinkles,null,Large pores,Texture uniformity,"CONCERN_TYPE=""Texture uniformity"";ZONE=[""Whole face"", ""Cheek""]"
DEMANDWARE,skin dr,b2f555eed98db96dec348677fa1123e7,2024-08-08T09:18:40Z,VIC,BRA,b65e462063ddb916ebfeca673211b966,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Male,null,null,null,null,No,null,Normal,wholeface,Wrinkles,null,Lack of radiance,Color uniformity,Whole face
DEMANDWARE,skin dr,b2f555eed98db96dec348677fa1123e7,2024-08-08T09:18:40Z,VIC,BRA,b65e462063ddb916ebfeca673211b966,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Male,null,null,null,null,No,null,Normal,wholeface,Large pores,null,Wrinkles,Texture uniformity,Whole face
DEMANDWARE,skin dr,b2f555eed98db96dec348677fa1123e7,2024-08-08T09:18:40Z,VIC,BRA,b65e462063ddb916ebfeca673211b966,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Male,null,null,null,null,No,null,Normal,wholeface,Large pores,null,Lack of firmness,Texture uniformity,"CONCERN_TYPE=""Texture uniformity"";ZONE=[""Whole face"", ""Lower part of the face""]"
DEMANDWARE,skin dr,b2f555eed98db96dec348677fa1123e7,2024-08-08T09:18:40Z,VIC,BRA,b65e462063ddb916ebfeca673211b966,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Male,null,null,null,null,No,null,Normal,wholeface,Large pores,null,Large pores,Texture uniformity,"CONCERN_TYPE=""Texture uniformity"";ZONE=[""Whole face"", ""Cheek""]"
DEMANDWARE,skin dr,b2f555eed98db96dec348677fa1123e7,2024-08-08T09:18:40Z,VIC,BRA,b65e462063ddb916ebfeca673211b966,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Male,null,null,null,null,No,null,Normal,wholeface,Large pores,null,Lack of radiance,Color uniformity,Whole face
DEMANDWARE,skin dr,b2f555eed98db96dec348677fa1123e7,2024-08-08T09:18:40Z,VIC,BRA,b65e462063ddb916ebfeca673211b966,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Male,null,null,null,null,No,null,Normal,wholeface,Lack of radiance,null,Wrinkles,Texture uniformity,Whole face
DEMANDWARE,skin dr,b2f555eed98db96dec348677fa1123e7,2024-08-08T09:18:40Z,VIC,BRA,b65e462063ddb916ebfeca673211b966,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Male,null,null,null,null,No,null,Normal,wholeface,Lack of radiance,null,Lack of firmness,Texture uniformity,"CONCERN_TYPE=""Textur

In [0]:
## DETALLE DE DUPLICADOS
tmp = spark.sql("""
select brand_mdm_id, count(brand_mdm_id) as counts
from def_gdm_vw
group by brand_mdm_id
having counts > 1
order by counts desc
--limit 1000
""")	

display(tmp)
tmp.createOrReplaceTempView("tmp_vw")

brand_mdm_id,counts
8473ba14591a9441ae2adf617f2af010,49
eca05292bb87bb742280e8f1a0c94453,36
e212502744122dfcafe7b55ddfc2313e,36
acf9e6df518eb0d7598323797f33a829,36
4b041bcee1a0b7db77d25f34d9488d34,36
5bacbc99b6c0750cca7eabde8770dd08,36
307d9b5aff99eb33db090340ee8b3fb7,36
2d9feb4510f4dfb9f132c1af8fc98dde,36
9c19849dae78f94ad7abdd419e5d96b5,36
c3225758d41b7d44e2cb826252fef02f,36


In [0]:
%sql
-- valida cuantos brand_mdm_id tienen mas de un registro
select counts, count(brand_mdm_id) as brand_mdm_id_counts
from tmp_vw
group by counts
order by brand_mdm_id_counts desc

counts,brand_mdm_id_counts
1,105880
36,1544
2,314
4,22
6,3
3,2
16,2
49,1


In [0]:
%sql
-- valida cuantos brand_mdm_id tienen mas de un registro
select registration_sub_source, count(distinct a.brand_mdm_id) as brand_mdm_id_counts
from def_gdm_vw a
inner join tmp_vw b
  on (a.brand_mdm_id = b.brand_mdm_id)
group by registration_sub_source
order by brand_mdm_id_counts desc

registration_sub_source,brand_mdm_id_counts
skin dr,1550
hair quiz,352


In [0]:
print(def_gdm.columns)

['source_name', 'registration_sub_source', 'brand_customer_id', 'brand_code', 'brand_country', 'brand_mdm_id', 'skin_tone', 'external_factor', 'hair_state', 'hair_texture', 'hair_type', 'scalp_type', 'hair_loss_volume', 'natural_hair_color', 'last_hair_style_look', 'hair_condition_frequency', 'last_hair_color_service', 'hair_color_service_frequency', 'scalp_sensitivity', 'hair_care_product_used', 'sun_exposure_frequency', 'created_date', 'gender', 'hair_color', 'category', 'product_category', 'hair_routine_heating_heating_tool', 'skin_sensitivity_skin_sensitivity', 'skin_sensitivity_zone', 'skin_type_skin_type', 'skin_type_zone', 'concern_improvement_goal_concern', 'channel_preference_product_purchase_channel', 'analysis_analysis_concern_concern', 'analysis_analysis_concern_concern_type', 'analysis_analysis_concern_zone']


In [0]:
%sql
-- valida cuantos brand_mdm_id tienen mas de un registro
select 
  source_name,
  brand_code,
  brand_country,
  registration_sub_source,
  'analysis_analysis_concern_zone' as attribute,
  analysis_analysis_concern_zone,
  count(distinct brand_mdm_id) as id_counts
from def_gdm_vw
group by all
order by source_name, brand_code, brand_country, registration_sub_source

source_name,brand_code,brand_country,registration_sub_source,attribute,analysis_analysis_concern_zone,id_counts
DEMANDWARE,DMC,BRA,skin dr,analysis_analysis_concern_zone,"CONCERN_TYPE=""Texture uniformity"";ZONE=[""Whole face"", ""Lower part of the face""]",1544
DEMANDWARE,DMC,BRA,skin dr,analysis_analysis_concern_zone,Whole face,1544
DEMANDWARE,DMC,BRA,skin dr,analysis_analysis_concern_zone,"CONCERN_TYPE=""Texture uniformity"";ZONE=[""Whole face"", ""Cheek""]",1544
DEMANDWARE,DMC,BRA,skin dr,analysis_analysis_concern_zone,null,18
DEMANDWARE,KER,COL,hair quiz,analysis_analysis_concern_zone,null,276
DEMANDWARE,KER,COL,hair quiz,analysis_analysis_concern_zone,Undereye,16
DEMANDWARE,KER,PER,hair quiz,analysis_analysis_concern_zone,null,93
DEMANDWARE,KER,PER,hair quiz,analysis_analysis_concern_zone,Undereye,7
DEMANDWARE,LRP,BRA,product finder,analysis_analysis_concern_zone,Whole face,34944
DEMANDWARE,LRP,BRA,product finder,analysis_analysis_concern_zone,null,17


In [0]:
tmp = def_gdm.drop('created_date','brand_mdm_id','brand_customer_id').distinct()
display(tmp)

source_name,registration_sub_source,brand_code,brand_country,skin_tone,external_factor,hair_state,hair_texture,hair_type,scalp_type,hair_loss_volume,natural_hair_color,last_hair_style_look,hair_condition_frequency,last_hair_color_service,hair_color_service_frequency,scalp_sensitivity,hair_care_product_used,sun_exposure_frequency,gender,hair_color,category,product_category,hair_routine_heating_heating_tool,skin_sensitivity_skin_sensitivity,skin_sensitivity_zone,skin_type_skin_type,skin_type_zone,concern_improvement_goal_concern,channel_preference_product_purchase_channel,analysis_analysis_concern_concern,analysis_analysis_concern_concern_type,analysis_analysis_concern_zone
JEBBIT,hair quiz,KER,COL,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Lack of shine,null,Lack of shine,Hair state,null
JEBBIT,hair quiz,KER,COL,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Frizzy,null,Frizzy,Hair state,null
JEBBIT,hair quiz,KER,COL,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Color fading,null,Color fading,Hair type,null
JEBBIT,hair quiz,KER,COL,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Damaged,null,Damaged,Hair type,null
JEBBIT,hair quiz,KER,COL,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Hair loss,null,Hair loss,Hair state,null
JEBBIT,hair quiz,KER,COL,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Dandruff,null,Dandruff,Scalp type,null
JEBBIT,hair quiz,KER,COL,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Undefined curls,null,Undefined curls,Hair state,null
JEBBIT,hair quiz,KER,COL,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Sensitivity,null,Sensitivity,Scalp type,null
JEBBIT,hair quiz,KER,COL,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Density,null,Density,Hair state,null
JEBBIT,hair quiz,KER,COL,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,Color fading,null,Damaged,Hair type,null


In [0]:
# Descomponer el array de fragrance_routine
tmp = gdm.select('fragrance_routine').\
  withColumn("exploded_fragrance", F.explode("fragrance_routine")).\
    select(
    "*",
    "exploded_fragrance.perfume_frequency",
    "exploded_fragrance.perfume_moment"
)

display(tmp.limit(10)) #internal_factor

fragrance_routine,exploded_fragrance,perfume_frequency,perfume_moment
"List(List(null, null))","List(null, null)",null,null
"List(List(null, null))","List(null, null)",null,null
"List(List(null, null))","List(null, null)",null,null
"List(List(null, null))","List(null, null)",null,null
"List(List(null, null))","List(null, null)",null,null
"List(List(null, null))","List(null, null)",null,null
"List(List(null, null))","List(null, null)",null,null
"List(List(null, null))","List(null, null)",null,null
"List(List(null, null))","List(null, null)",null,null
"List(List(null, null))","List(null, null)",null,null


In [0]:
# Descomponer el array de fragrance_routine
tmp = gdm.select('dept_store_regional').\
  withColumn("exploded_internal_factor", F.explode_outer("dept_store_regional"))

display(tmp.limit(10)) #internal_factor

dept_store_regional,exploded_internal_factor
null,null
null,null
null,null
null,null
null,null
null,null
null,null
null,null
null,null
null,null
